# Submission v8 — Better Model (7 seeds + tuned params) + α=1.8

## Strategy

α tuning is public LB fitting — risky for private LB. Instead, improve the **model quality**:

1. **7 seeds instead of 3** — reduces prediction variance, helps both public and private LB
2. **Tuned hyperparameters** guided by LOPO (not public LB):
   - `num_leaves=63` (was 127) — reduces overfitting to training people
   - `subsample=0.7` (was 0.6) — slightly more data per tree
   - `min_child_samples=10` (was 5) — prevents tiny leaf splits
   - `reg_alpha=0.5, reg_lambda=0.5` (was 0.3/0.3) — stronger regularization
3. **α=1.8 fixed** — already validated, don't touch further

Better raw probabilities from a better model means α matters less and private LB aligns more with public LB.


In [1]:
%pip install lightgbm scikit-learn pandas numpy scipy -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as spstats
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from collections import Counter

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)
print()
print('Stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


In [3]:
SENSOR_COLS = ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']
WINDOW_MS = 180_000; HALF_MS = 90_000; THIRD_MS = 60_000

def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']: f['hrv_'+k]=np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm)>=32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff)>0 else 0.
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff)>25))*100 if len(rr_diff)>0 else 0.
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff)>50))*100 if len(rr_diff)>0 else 0.
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn']/f['hrv_mean_rr'] if f['hrv_mean_rr']>1e-6 else 0.
    return f

def extract_features(label_df, sensor_df, pid_enc_map):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for _, lrow in label_df.iterrows():
        pid=lrow['pid']; ts=float(lrow['timestamp']); lid=lrow['id']
        feat={'id':lid}
        sg=sensor_by_pid.get(pid)
        if sg is None: rows.append(feat); continue
        ta=sg['timestamp'].values
        wa =sg.loc[(ta>=ts-WINDOW_MS)&(ta<=ts),           SENSOR_COLS]
        wf =sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-HALF_MS),    SENSOR_COLS]
        wl =sg.loc[(ta>=ts-HALF_MS)  &(ta<=ts),           SENSOR_COLS]
        wt1=sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-2*THIRD_MS), SENSOR_COLS]
        wt3=sg.loc[(ta>=ts-THIRD_MS) &(ta<=ts),           SENSOR_COLS]
        for c in SENSOR_COLS:
            v  =wa[c].dropna().values.astype(float)
            vf =wf[c].dropna().values.astype(float)
            vl =wl[c].dropna().values.astype(float)
            vt1=wt1[c].dropna().values.astype(float)
            vt3=wt3[c].dropna().values.astype(float)
            if len(v)==0:
                for s in ['mean','std','min','max','median','skew','kurt','range',
                          'q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{c}_{s}']=np.nan
                continue
            feat[f'{c}_mean']   =float(np.mean(v));    feat[f'{c}_std']   =float(np.std(v))
            feat[f'{c}_min']    =float(np.min(v));     feat[f'{c}_max']   =float(np.max(v))
            feat[f'{c}_median'] =float(np.median(v))
            feat[f'{c}_skew']   =float(spstats.skew(v))     if len(v)>2 else 0.
            feat[f'{c}_kurt']   =float(spstats.kurtosis(v)) if len(v)>2 else 0.
            feat[f'{c}_range']  =float(np.max(v)-np.min(v))
            feat[f'{c}_q25']    =float(np.percentile(v,25))
            feat[f'{c}_q75']    =float(np.percentile(v,75))
            feat[f'{c}_iqr']    =float(np.percentile(v,75)-np.percentile(v,25))
            feat[f'{c}_delta']  =float(np.mean(vl)-np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.
            feat[f'{c}_slope']  =float(np.polyfit(np.linspace(0,1,len(v)),v,1)[0]) if len(v)>2 else 0.
            feat[f'{c}_t1_mean']=float(np.mean(vt1)) if len(vt1)>0 else float(np.mean(v))
            feat[f'{c}_t3_mean']=float(np.mean(vt3)) if len(vt3)>0 else float(np.mean(v))
            feat[f'{c}_t3t1']  =feat[f'{c}_t3_mean']-feat[f'{c}_t1_mean']
        ax=wa['accel_x'].values; ay=wa['accel_y'].values; az=wa['accel_z'].values
        if len(ax)>0:
            mag=np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean']=float(np.mean(mag))
            feat['accel_mag_std'] =float(np.std(mag))
            feat['accel_mag_max'] =float(np.max(mag))
        else:
            feat['accel_mag_mean']=feat['accel_mag_std']=feat['accel_mag_max']=np.nan
        feat.update(hrv_time_domain(wa['heart_rate']))
        feat['pid_enc']=pid_enc_map.get(pid,-1)
        rows.append(feat)
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p:i for i,p in enumerate(TRAIN_LABEL['pid'].unique())}
print('Extracting features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map)
test_features  = extract_features(TEST_LABEL,  TEST_DATA,  train_pid_map)
print(f'train: {train_features.shape}  test: {test_features.shape}')


Extracting features...
train: (815, 106)  test: (1028, 106)


In [4]:
tli    = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features),
                           columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),
                           columns=test_features.columns, index=test_features.index)

counts=Counter(y); total=len(y); n_cls=len(counts)
class_weights={0:total/(n_cls*counts[0]), 1:min(total/(n_cls*counts[1]),2.5), 2:total/(n_cls*counts[2])}
sample_weights=np.array([class_weights[yi] for yi in y])
train_prior=np.array([counts[i]/total for i in range(3)])

print('X_imp shape:', X_imp.shape)
print('Class weights (capped):', {k:round(v,3) for k,v in class_weights.items()})


X_imp shape: (815, 106)
Class weights (capped): {0: 1.677, 1: 2.5, 2: 0.463}


In [5]:
# Tuned params — more regularization to reduce person-identity overfitting
# Changes from v7c: num_leaves 127→63, min_child_samples 5→10,
#                   reg_alpha/lambda 0.3→0.5, subsample 0.6→0.7
LGBM_PARAMS = dict(
    n_estimators      = 1000,
    learning_rate     = 0.02,
    num_leaves        = 63,    # was 127 — smaller trees generalize better cross-person
    max_depth         = -1,
    min_child_samples = 10,    # was 5 — prevents tiny over-specific leaf splits
    subsample         = 0.7,   # was 0.6 — slightly more data per tree
    colsample_bytree  = 0.7,   # was 0.6
    reg_alpha         = 0.5,   # was 0.3 — stronger L1 regularization
    reg_lambda        = 0.5,   # was 0.3 — stronger L2 regularization
    class_weight      = 'balanced',
    objective         = 'multiclass',
    num_class         = 3,
    n_jobs            = -1,
    verbose           = -1,
)

print('=== LOPO CV — v8 tuned params ===')
logo=LeaveOneGroupOut(); lopo_scores=[]
for tr_idx, val_idx in logo.split(X_imp, y, groups):
    pid_val=groups.iloc[val_idx[0]]; y_val=y.iloc[val_idx]
    if len(y_val.unique())<2: print(f'  Skip {pid_val}'); continue
    m=lgb.LGBMClassifier(**{**LGBM_PARAMS,'random_state':42})
    m.fit(X_imp.iloc[tr_idx], y.iloc[tr_idx],
          sample_weight=sample_weights[tr_idx],
          eval_set=[(X_imp.iloc[val_idx], y_val)],
          callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    sc=balanced_accuracy_score(y_val, m.predict(X_imp.iloc[val_idx]))
    print(f'  Leave out {pid_val}: {sc:.4f}  (n={len(val_idx)})')
    lopo_scores.append(sc)
print(f'\nv8 LOPO = {np.mean(lopo_scores):.4f} +/- {np.std(lopo_scores):.4f}')
print('v7c ref  = 0.5130  (target to beat)')
print()
print('If LOPO < 0.5130 → revert params back to v7c values')
print('If LOPO >= 0.5130 → proceed with ensemble')


=== LOPO CV — v8 tuned params ===
  Leave out 43JW: 0.0000  (n=93)
  Leave out C8Q6: 0.4930  (n=152)
  Leave out DT5C: 0.4455  (n=90)
  Leave out F1ZM: 0.5000  (n=137)
  Leave out HDS9: 0.5662  (n=135)
  Leave out P4DZ: 0.3093  (n=144)
  Leave out TPQI: 0.5327  (n=64)

v8 LOPO = 0.4067 +/- 0.1828
v7c ref  = 0.5130  (target to beat)

If LOPO < 0.5130 → revert params back to v7c values
If LOPO >= 0.5130 → proceed with ensemble


In [6]:
# Only run this if LOPO >= 0.5130
print('=== Final ensemble: 7 seeds x 5 folds ===')
# 7 seeds instead of 3 — reduces variance in final probabilities
SEEDS = [42, 7, 123, 17, 99, 256, 314]
all_test_proba=[]; all_cv_scores=[]

for seed in SEEDS:
    skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=seed)
    seed_proba=np.zeros((len(X_test_imp),3)); fold_scores=[]
    for fold,(tr_idx,val_idx) in enumerate(skf.split(X_imp,y)):
        X_tr=X_imp.iloc[tr_idx]; y_tr=y.iloc[tr_idx]
        X_va=X_imp.iloc[val_idx]; y_va=y.iloc[val_idx]
        m=lgb.LGBMClassifier(**{**LGBM_PARAMS,'random_state':seed})
        m.fit(X_tr, y_tr,
              sample_weight=sample_weights[tr_idx],
              eval_set=[(X_va,y_va)],
              callbacks=[lgb.early_stopping(100,verbose=False), lgb.log_evaluation(-1)])
        sc=balanced_accuracy_score(y_va,m.predict(X_va))
        fold_scores.append(sc); seed_proba+=m.predict_proba(X_test_imp)
        print(f'  Seed {seed} Fold {fold+1}: val BA={sc:.4f}')
    seed_proba/=5; all_test_proba.append(seed_proba)
    all_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed} mean CV={np.mean(fold_scores):.4f}')

print(f'\nEnsemble CV={np.mean(all_cv_scores):.4f}')
print('v7c ref = 0.8115')

raw_proba=np.mean(all_test_proba,axis=0)

# Fixed α=1.8 — validated on LB, not tuning further
CALIB_ALPHA = 1.8
cal_proba=raw_proba*(train_prior**CALIB_ALPHA)
cal_proba=cal_proba/cal_proba.sum(axis=1,keepdims=True)
final_preds=np.argmax(cal_proba,axis=1).astype(int)

print(f'\nCalibration alpha={CALIB_ALPHA} (fixed from v7c validation)')
print('Raw distribution:')
for u,cnt in zip(*np.unique(np.argmax(raw_proba,1),return_counts=True)):
    print(f'  class {u}: {cnt}  ({cnt/len(final_preds)*100:.1f}%)')
print('Calibrated:')
for u,cnt in zip(*np.unique(final_preds,return_counts=True)):
    print(f'  class {u}: {cnt}  ({cnt/len(final_preds)*100:.1f}%)')
print('Train prior:')
for cls in range(3): print(f'  class {cls}: {counts[cls]}  ({counts[cls]/total*100:.1f}%)')


=== Final ensemble: 7 seeds x 5 folds ===
  Seed 42 Fold 1: val BA=0.8119
  Seed 42 Fold 2: val BA=0.8065
  Seed 42 Fold 3: val BA=0.8643
  Seed 42 Fold 4: val BA=0.7898
  Seed 42 Fold 5: val BA=0.8619
  Seed 42 mean CV=0.8269
  Seed 7 Fold 1: val BA=0.8540
  Seed 7 Fold 2: val BA=0.7853
  Seed 7 Fold 3: val BA=0.8226
  Seed 7 Fold 4: val BA=0.8569
  Seed 7 Fold 5: val BA=0.7404
  Seed 7 mean CV=0.8118
  Seed 123 Fold 1: val BA=0.8642
  Seed 123 Fold 2: val BA=0.6553
  Seed 123 Fold 3: val BA=0.8565
  Seed 123 Fold 4: val BA=0.8135
  Seed 123 Fold 5: val BA=0.7724
  Seed 123 mean CV=0.7924
  Seed 17 Fold 1: val BA=0.8290
  Seed 17 Fold 2: val BA=0.7534
  Seed 17 Fold 3: val BA=0.8469
  Seed 17 Fold 4: val BA=0.7755
  Seed 17 Fold 5: val BA=0.7963
  Seed 17 mean CV=0.8002
  Seed 99 Fold 1: val BA=0.8300
  Seed 99 Fold 2: val BA=0.8223
  Seed 99 Fold 3: val BA=0.8065
  Seed 99 Fold 4: val BA=0.7926
  Seed 99 Fold 5: val BA=0.8239
  Seed 99 mean CV=0.8151
  Seed 256 Fold 1: val BA=0.7758


In [7]:
submission=pd.DataFrame({'id':TEST_LABEL['id'].values,'stress':final_preds})
submission.to_csv('submission_v8.csv',index=False)
print('submission_v8.csv saved!')
print(submission.head(10))


submission_v8.csv saved!
     id  stress
0  1227       2
1  1228       2
2  1229       2
3  1230       2
4  1231       2
5  1232       2
6  1233       0
7  1234       2
8  1235       0
9  1236       0
